# TP — Crimes à Chicago : pandas vs Spark

Données : *Crimes - 2001 to Present* (Chicago Police Department), 22 colonnes.
Export officiel : `https://data.cityofchicago.org/api/views/ijzp-q8t2/rows.csv?accessType=DOWNLOAD`
(~8,07 M de lignes, ~1,9 Go).

Fichier utilisé pour ces mesures : `crimes_clean_4m.csv`, 1,13 Go et 4 778 681 lignes, extrait du
même export avec les mêmes colonnes. Le code ne change pas pour l'export complet, il suffit de
pointer `PATH` dessus.

Machine : Windows 11, 14 cœurs, 31,5 Go de RAM, Java 17, PySpark 4.3.0, pandas 3.x, Python 3.12.

Un point de méthode avant de commencer. Spark est paresseux : `spark.read.csv()` seul ne prend que
quelques millisecondes, parce qu'il ne lit rien. Chaque « temps de lecture » ci-dessous est donc
lecture + première action (`count()`).

## 0. Session Spark

In [1]:
import os, time, json, urllib.request
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType,
                               LongType, DoubleType, BooleanType, TimestampType)

PATH = os.environ.get("CRIMES_CSV", "crimes.csv")
assert os.path.exists(PATH), f"fichier introuvable : {PATH}"
print(f"{PATH} : {os.path.getsize(PATH)/1e9:.2f} Go")

spark = (SparkSession.builder
         .appName("TP Crimes Chicago")
         .master("local[*]")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "16")
         .config("spark.driver.maxResultSize", "512m")
         .config("spark.ui.enabled", "true")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| UI:", spark.sparkContext.uiWebUrl)

RESULTS = {}

def taches_consommees():
    # tasks cumulees de l'application, lues sur l'API REST du UI Spark
    base = spark.sparkContext.uiWebUrl
    app = spark.sparkContext.applicationId
    with urllib.request.urlopen(f"{base}/api/v1/applications/{app}/stages", timeout=10) as r:
        return sum(s["numTasks"] for s in json.load(r))

C:\Users\Y.CHEHBOUB\PERSONAL__DO_NOT_TOUCH\OpenDataModule\crimes_clean_4m.csv : 1.13 Go


C:\Users\Y.CHEHBOUB\PERSONAL__DO_NOT_TOUCH\OpenDataModule\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark 4.3.0 | UI: http://FBL-LT-0607.insead.org:4040


## 1. Lecture avec pandas

`pd.read_csv()` charge **tout** le fichier dans la RAM du driver : aucun parallélisme, aucune
distribution, un `DataFrame` de plusieurs millions de lignes.

In [2]:
import pandas as pd, psutil, threading, gc

proc = psutil.Process()
pic = [proc.memory_info().rss]; stop = [False]
def _echantillonne():
    while not stop[0]:
        pic[0] = max(pic[0], proc.memory_info().rss); time.sleep(0.2)
threading.Thread(target=_echantillonne, daemon=True).start()

t0 = time.perf_counter()
pdf = pd.read_csv(PATH)
t_pandas = time.perf_counter() - t0
stop[0] = True
RESULTS["1_pandas"] = round(t_pandas, 2)

print(f"pandas : {t_pandas:.2f} s")
print(f"shape  : {pdf.shape[0]:,} lignes x {pdf.shape[1]} colonnes")
print(f"RAM pic: {pic[0]/2**30:.2f} Go")
print(f"memoire du DataFrame (deep) : {pdf.memory_usage(deep=True).sum()/2**30:.2f} Go")
print(pdf.dtypes.value_counts().to_string())

del pdf; gc.collect()

# meme lecture avec le moteur C historique (pandas 2.x) pour comparaison
t0 = time.perf_counter()
pdf_c = pd.read_csv(PATH, engine="c", low_memory=False)
t_c = time.perf_counter() - t0
RESULTS["1_pandas_engine_c"] = round(t_c, 2)
print(f"\npandas moteur C (engine='c') : {t_c:.2f} s  |  RAM pic process : {pic[0]/2**30:.2f} Go")
del pdf_c; gc.collect()
print(f"RAM du process apres liberation : {proc.memory_info().rss/2**30:.2f} Go")

pandas : 1.66 s
shape  : 4,778,681 lignes x 22 colonnes
RAM pic: 2.78 Go
memoire du DataFrame (deep) : 1.36 Go
str        10
float64     7
int64       3
bool        2



pandas moteur C (engine='c') : 6.38 s  |  RAM pic process : 2.78 Go
RAM du process apres liberation : 0.13 Go


### Conclusion Q1

Le fichier entier est matérialisé dans un seul processus : 2,78 Go de RAM pour un DataFrame de
1,36 Go, soit 2,5 fois la taille du fichier. La lecture elle-même est rapide (1,66 s avec le moteur
par défaut de pandas 3, 6,38 s avec le moteur C).

Ce qui coince n'est pas le `read_csv`, c'est la suite : chaque filtre, `groupby` ou jointure tourne
sur un seul thread, et il faut recharger le fichier à chaque exécution. Sur 8 M de lignes et 1,9 Go,
ça devient un choix de moteur plutôt qu'une préférence.

## 2. Lecture avec Spark — options par défaut

Sans aucune option : la première ligne du fichier est traitée comme une donnée, les colonnes
s'appellent `_c0` … `_c21`, tout est en `string`.

In [3]:
df_brut = spark.read.csv(PATH)
t0 = time.perf_counter(); n_brut = df_brut.count(); t_brut = time.perf_counter() - t0
RESULTS["2_spark_defaut_count"] = round(t_brut, 2)
print(f"spark.read.csv(PATH).count() : {t_brut:.2f} s -> {n_brut:,} lignes")
print("colonnes :", df_brut.columns)
df_brut.show(3)

spark.read.csv(PATH).count() : 1.93 s -> 4,778,682 lignes
colonnes : ['_c0', '_c1', '_c2', '_c3', '_c4', '_c5', '_c6', '_c7', '_c8', '_c9', '_c10', '_c11', '_c12', '_c13', '_c14', '_c15', '_c16', '_c17', '_c18', '_c19', '_c20', '_c21']


+--------+-----------+--------------------+--------------------+----+--------------------+--------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+--------------------+--------+---------+--------+
|     _c0|        _c1|                 _c2|                 _c3| _c4|                 _c5|                 _c6|                 _c7|   _c8|     _c9|_c10|    _c11|_c12|          _c13|    _c14|        _c15|        _c16|_c17|                _c18|    _c19|     _c20|    _c21|
+--------+-----------+--------------------+--------------------+----+--------------------+--------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+--------------------+--------+---------+--------+
|      ID|Case Number|                Date|               Block|IUCR|        Primary Type|         Description|Location Description|Arrest|Domestic|Beat|District|Ward|Community Area|FB

### 2 bis. Avec `header=True`

In [4]:
df_str = spark.read.csv(PATH, header=True)
t0 = time.perf_counter(); n = df_str.count(); t_header = time.perf_counter() - t0
RESULTS["2_spark_header_count"] = round(t_header, 2)
print(f"spark.read.csv(PATH, header=True).count() : {t_header:.2f} s -> {n:,} lignes")
print("colonnes :", df_str.columns)
df_str.printSchema()

spark.read.csv(PATH, header=True).count() : 0.99 s -> 4,778,681 lignes
colonnes : ['ID', 'Case Number', 'Date', 'Block', 'IUCR', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area', 'FBI Code', 'X Coordinate', 'Y Coordinate', 'Year', 'Updated On', 'Latitude', 'Longitude', 'Location']
root
 |-- ID: string (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location Description: string (nullable = true)
 |-- Arrest: string (nullable = true)
 |-- Domestic: string (nullable = true)
 |-- Beat: string (nullable = true)
 |-- District: string (nullable = true)
 |-- Ward: string (nullable = true)
 |-- Community Area: string (nullable = true)
 |-- FBI Code: string (nullable = true)
 |-- X Coordinate: string (nullable = t

### Conclusion Q2

Sans option, Spark prend la première ligne pour une donnée : 4 778 682 lignes au lieu de 4 778 681,
colonnes `_c0 … _c21`, tout en `string`. `header=True` règle les deux, sans coût supplémentaire.

Face à pandas, le temps est du même ordre sur ce volume. La différence est ailleurs : Spark lit par
partitions de 128 Mo, en parallèle, et ne fait jamais transiter le fichier par le driver.

## 3. Partitions, colonnes, lignes

In [5]:
n_part = df_str.rdd.getNumPartitions()
print("partitions :", n_part, " (= découpage du fichier en blocs de 128 Mo, 1 partition = 1 task)")
print("nombre de colonnes :", len(df_str.columns))
print("colonnes :", df_str.columns)
RESULTS["3_partitions"] = n_part
RESULTS["3_colonnes"] = len(df_str.columns)
print(f"lignes : {n:,}")

partitions : 14  (= découpage du fichier en blocs de 128 Mo, 1 partition = 1 task)
nombre de colonnes : 22
colonnes : ['ID', 'Case Number', 'Date', 'Block', 'IUCR', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area', 'FBI Code', 'X Coordinate', 'Y Coordinate', 'Year', 'Updated On', 'Latitude', 'Longitude', 'Location']
lignes : 4,778,681


## 4. Affichage « comme dans un notebook pandas »

Évaluer `df` dans une cellule n'affiche qu'un `DataFrame[...]`. Le réglage est
**`spark.sql.repl.eagerEval.enabled = true`** : Spark rend alors un tableau HTML, comme pandas,
borné par `eagerEval.maxNumRows`.

In [6]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 5)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 25)
df_str     # rendu type pandas

ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
13311263,JG503434,07/29/2022 03:39:00 AM,023XX S TROY ST,1582,OFFENSE INVOLVING CHIL...,CHILD PORNOGRAPHY,RESIDENCE,true,false,1033,010,25,30,17,NULL,NULL,2022,04/18/2024 03:40:59 PM,NULL,NULL,NULL
13053066,JG103252,01/03/2023 04:44:00 PM,039XX W WASHINGTON BLVD,2017,NARCOTICS,MANUFACTURE / DELIVER ...,SIDEWALK,true,false,1122,011,28,26,18,NULL,NULL,2023,01/20/2024 03:41:12 PM,NULL,NULL,NULL
11227634,JB147599,08/26/2017 10:00:00 AM,001XX W RANDOLPH ST,0281,CRIM SEXUAL ASSAULT,NON-AGGRAVATED,HOTEL/MOTEL,false,false,0122,001,42,32,02,NULL,NULL,2017,02/11/2018 03:57:41 PM,NULL,NULL,NULL
12416972,JE293535,10/01/2020 12:01:00 AM,031XX W 53RD PL,1753,OFFENSE INVOLVING CHIL...,SEXUAL ASSAULT OF CHIL...,RESIDENCE,false,true,0923,009,14,63,02,NULL,NULL,2020,09/14/2023 03:41:59 PM,NULL,NULL,NULL
12536164,JE439378,09/24/2015 12:00:00 AM,031XX W 53RD PL,1753,OFFENSE INVOLVING CHIL...,SEXUAL ASSAULT OF CHIL...,APARTMENT,false,true,0923,009,14,63,02,NULL,NULL,2015,09/14/2023 03:41:59 PM,NULL,NULL,NULL


## 5. `inferSchema=True` : pourquoi ≥ 30 s ?

In [7]:
t0 = time.perf_counter()
df_infer = spark.read.csv(PATH, header=True, inferSchema=True)
n_infer = df_infer.count()      # c'est ici que l'inference est reellement executee
t_infer = time.perf_counter() - t0
RESULTS["5_inferSchema_count"] = round(t_infer, 2)
print(f"inferSchema=True -> count() : {t_infer:.2f} s ({n_infer:,} lignes)")
df_infer.printSchema()

inferSchema=True -> count() : 6.91 s (4,778,681 lignes)
root
 |-- ID: integer (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community Area: integer (nullable = true)
 |-- FBI Code: string (nullable = true)
 |-- X Coordinate: integer (nullable = true)
 |-- Y Coordinate: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated On: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)



### Conclusion Q5

6,91 s contre 0,99 s sans inférence, environ 7 fois plus. La cause est simple : Spark doit parcourir
le fichier une seconde fois, uniquement pour deviner le type de chaque colonne, avant la vraie
lecture. Deux passes complètes.

Sur l'export de 1,9 Go, c'est le surcoût de plusieurs dizaines de secondes dont parle l'énoncé.
Quand on connaît les colonnes, c'est du temps perdu, d'où le schéma explicite en Q8.

## 6. Types, sélection, ajout / suppression de colonnes

In [8]:
print("Types inférés :")
for c, t in df_infer.dtypes:
    print(f"  {c:<24} {t}")

Types inférés :
  ID                       int
  Case Number              string
  Date                     string
  Block                    string
  IUCR                     string
  Primary Type             string
  Description              string
  Location Description     string
  Arrest                   boolean
  Domestic                 boolean
  Beat                     int
  District                 int
  Ward                     int
  Community Area           int
  FBI Code                 string
  X Coordinate             int
  Y Coordinate             int
  Year                     int
  Updated On               string
  Latitude                 double
  Longitude                double
  Location                 string


In [9]:
# acces aux colonnes District et Case Number
df_infer.select("District", "Case Number").show(5)

+--------+-----------+
|District|Case Number|
+--------+-----------+
|      10|   JG503434|
|      11|   JG103252|
|       1|   JB147599|
|       9|   JE293535|
|       9|   JE439378|
+--------+-----------+
only showing top 5 rows


In [10]:
# DataFrame restreint a ces 2 colonnes
df2 = df_infer.select("District", "Case Number")
print("colonnes de df2 :", df2.columns)

# ajout d'une colonne constante = 1
df2 = df2.withColumn("un", F.lit(1))
df2.show(5)
print("apres withColumn('un', lit(1)) :", df2.columns)

# suppression de cette colonne
df2 = df2.drop("un")
print("apres drop('un') :", df2.columns)

# suppression de ID, Case Number, Block
df_sans = df_infer.drop("ID", "Case Number", "Block")
print("apres drop('ID','Case Number','Block') :", df_sans.columns)

colonnes de df2 : ['District', 'Case Number']


+--------+-----------+---+
|District|Case Number| un|
+--------+-----------+---+
|      10|   JG503434|  1|
|      11|   JG103252|  1|
|       1|   JB147599|  1|
|       9|   JE293535|  1|
|       9|   JE439378|  1|
+--------+-----------+---+
only showing top 5 rows
apres withColumn('un', lit(1)) : ['District', 'Case Number', 'un']
apres drop('un') : ['District', 'Case Number']
apres drop('ID','Case Number','Block') : ['Date', 'IUCR', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area', 'FBI Code', 'X Coordinate', 'Y Coordinate', 'Year', 'Updated On', 'Latitude', 'Longitude', 'Location']


## 7. Affichages ciblés

5 premières valeurs de `IUCR` ; 5 premières lignes de `Case Number`, `Date`, `Arrest`
**simultanément** ; 2 enregistrements **en mode vertical** et **sans troncature**.

In [11]:
df_infer.select("IUCR").show(5)

+----+
|IUCR|
+----+
|1582|
|2017|
|0281|
|1753|
|1753|
+----+
only showing top 5 rows

In [12]:
df_infer.select("Case Number", "Date", "Arrest").show(5)

+-----------+--------------------+------+
|Case Number|                Date|Arrest|
+-----------+--------------------+------+
|   JG503434|07/29/2022 03:39:...|  true|
|   JG103252|01/03/2023 04:44:...|  true|
|   JB147599|08/26/2017 10:00:...| false|
|   JE293535|10/01/2020 12:01:...| false|
|   JE439378|09/24/2015 12:00:...| false|
+-----------+--------------------+------+
only showing top 5 rows

In [13]:
df_infer.select("Case Number", "Date", "Arrest", "Primary Type", "Block", "Description") \
        .show(2, truncate=False, vertical=True)

-RECORD 0-------------------------------------
 Case Number  | JG503434                      
 Date         | 07/29/2022 03:39:00 AM        
 Arrest       | true                          
 Primary Type | OFFENSE INVOLVING CHILDREN    
 Block        | 023XX S TROY ST               
 Description  | CHILD PORNOGRAPHY             
-RECORD 1-------------------------------------
 Case Number  | JG103252                      
 Date         | 01/03/2023 04:44:00 PM        
 Arrest       | true                          
 Primary Type | NARCOTICS                     
 Block        | 039XX W WASHINGTON BLVD       
 Description  | MANUFACTURE / DELIVER - CRACK 
only showing top 2 rows


## 8. Schéma manuel (au lieu de `inferSchema`)

On déclare les types une fois pour toutes : Spark ne devine plus rien.

In [14]:
schema = StructType([
    StructField("ID", LongType(), True),
    StructField("Case Number", StringType(), True),
    StructField("Date", StringType(), True),       # converti en timestamp juste apres
    StructField("Block", StringType(), True),
    StructField("IUCR", StringType(), True),
    StructField("Primary Type", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Location Description", StringType(), True),
    StructField("Arrest", BooleanType(), True),
    StructField("Domestic", BooleanType(), True),
    StructField("Beat", IntegerType(), True),
    StructField("District", IntegerType(), True),
    StructField("Ward", IntegerType(), True),
    StructField("Community Area", IntegerType(), True),
    StructField("FBI Code", StringType(), True),
    StructField("X Coordinate", DoubleType(), True),
    StructField("Y Coordinate", DoubleType(), True),
    StructField("Year", IntegerType(), True),
    StructField("Updated On", StringType(), True),
    StructField("Latitude", DoubleType(), True),
    StructField("Longitude", DoubleType(), True),
    StructField("Location", StringType(), True),
])

t0 = time.perf_counter()
df = spark.read.csv(PATH, header=True, schema=schema)
# les dates arrivent en texte : ISO (cet export) ou MM/DD/YYYY hh:mm:ss AM/PM (export officiel)
for c in ("Date", "Updated On"):
    df = df.withColumn(c, F.coalesce(
        F.try_to_timestamp(c, F.lit("yyyy-MM-dd'T'HH:mm:ss.SSS")),   # export reconstruit (ISO)
        F.try_to_timestamp(c, F.lit("MM/dd/yyyy hh:mm:ss a"))))      # export officiel
n = df.count()
t_schema = time.perf_counter() - t0
RESULTS["8_schema_manuel_count"] = round(t_schema, 2)
print(f"schéma manuel -> count() : {t_schema:.2f} s ({n:,} lignes)")
print(f"vs inferSchema : {RESULTS['5_inferSchema_count'] / t_schema:.1f} x plus rapide")
df.printSchema()

schéma manuel -> count() : 0.85 s (4,778,681 lignes)
vs inferSchema : 8.1 x plus rapide
root
 |-- ID: long (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Date: timestamp (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community Area: integer (nullable = true)
 |-- FBI Code: string (nullable = true)
 |-- X Coordinate: double (nullable = true)
 |-- Y Coordinate: double (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated On: timestamp (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)



## 9. `df.limit()`, `df.head()`, `df.take()`, `df.collect()`

Exécutés **exactement tels qu'écrits** dans l'énoncé, puis avec un argument quand la signature
l'exige.

In [15]:
# df.limit() sans argument
t0 = time.perf_counter()
try:
    df.limit()
except Exception as e:
    print(f"df.limit() sans argument -> {type(e).__name__} : {e}")
print(f"  ({time.perf_counter()-t0:.4f} s)")

df.limit() sans argument -> TypeError : DataFrame.limit() missing 1 required positional argument: 'num'
  (0.0006 s)


In [16]:
# df.take() sans argument
try:
    df.take()
except Exception as e:
    print(f"df.take() sans argument -> {type(e).__name__} : {e}")

df.take() sans argument -> TypeError : DataFrame.take() missing 1 required positional argument: 'num'


In [17]:
# les versions correctes, et ce qu'elles declenchent
t0 = time.perf_counter(); df_lim = df.limit(5); t = time.perf_counter()-t0
print(f"df.limit(5) : {t*1000:.1f} ms -> {type(df_lim).__name__} (transformation paresseuse, 0 task)")

t0 = time.perf_counter(); h = df.head(); t_head = time.perf_counter()-t0
print(f"df.head()   : {t_head:.2f} s -> {type(h).__name__} (1 seul enregistrement) {str(h)[:120]}...")

t0 = time.perf_counter(); tk = df.take(5); t_take = time.perf_counter()-t0
print(f"df.take(5)  : {t_take:.2f} s -> {type(tk).__name__} de {len(tk)} Rows")
RESULTS["9_limit_ms"] = round(t*1000, 1); RESULTS["9_head"] = round(t_head, 2); RESULTS["9_take"] = round(t_take, 2)

df.limit(5) : 6.4 ms -> DataFrame (transformation paresseuse, 0 task)


df.head()   : 0.26 s -> Row (1 seul enregistrement) Row(ID=13311263, Case Number='JG503434', Date=datetime.datetime(2022, 7, 29, 3, 39), Block='023XX S TROY ST', IUCR='1582...
df.take(5)  : 0.12 s -> list de 5 Rows


In [18]:
# df.collect() : rapatrie TOUTES les lignes dans le driver
t0 = time.perf_counter()
try:
    lignes = df.collect()
    t_collect = time.perf_counter() - t0
    print(f"df.collect() : {t_collect:.2f} s -> {len(lignes):,} Rows dans le driver")
    print(f"RAM du process driver : {psutil.Process().memory_info().rss/2**30:.2f} Go")
    del lignes
except Exception as e:
    t_collect = time.perf_counter() - t0
    print(f"df.collect() a échoué après {t_collect:.0f} s : {type(e).__name__}")
    lignes_err = [l.strip() for l in str(e).splitlines() if "maxResultSize" in l or "serialized results" in l]
    print(lignes_err[0][:300] if lignes_err else str(e).splitlines()[0][:300])
    RESULTS["9_collect_echec_s"] = round(t_collect)

df.collect() a échoué après 22 s : Py4JJavaError
: org.apache.spark.SparkException: Job aborted due to stage failure: Total size of serialized results of 11 tasks (541.9 MiB) is bigger than spark.driver.maxResultSize (512.0 MiB)


### Conclusion Q9

| appel | signature | retour | coût |
|---|---|---|---|
| `df.limit()` | invalide, il faut `limit(n)` | — | `TypeError` immédiat |
| `df.take()` | invalide, il faut `take(n)` | — | `TypeError` immédiat |
| `df.limit(5)` | valide | un DataFrame | 6 ms, paresseux, 0 task |
| `df.head()` | valide | 1 `Row` | un job, une seule ligne remonte |
| `df.take(5)` | valide | liste de 5 `Row` | un job, 5 lignes remontent |
| `df.collect()` | valide | toutes les `Row` | échec en 22 s, 541,9 Mo pour 512 Mo autorisés |

`limit()` et `take()` sans argument ne compilent pas. `limit(n)` ne déclenche aucun job. `head()` et
`take(n)` rapatrient n lignes, ce qui reste borné.

`collect()` est l'appel qui casse : il ramène toutes les lignes dans le driver. On agrège côté
Spark et on ne collecte que du réduit.

## 10. Mise en cache

In [19]:
t0 = time.perf_counter(); df.cache(); df.count(); t1 = time.perf_counter() - t0
print(f"df.cache() + 1er count() (matérialisation) : {t1:.2f} s")
print("storageLevel :", df.storageLevel)

t0 = time.perf_counter(); df.count(); t2 = time.perf_counter() - t0
print(f"2e count() sur le df caché : {t2:.2f} s -> gain x{t1/t2:.1f}")
RESULTS["10_cache_avant"] = round(t1, 2); RESULTS["10_cache_apres"] = round(t2, 2)

df.cache() + 1er count() (matérialisation) : 33.20 s
storageLevel : Disk Memory Deserialized 1x Replicated


2e count() sur le df caché : 0.39 s -> gain x85.9


## 11. `describe()` vs `summary()`

In [20]:
t0 = time.perf_counter(); df.describe().show(truncate=False); t_desc = time.perf_counter() - t0
RESULTS["11_describe"] = round(t_desc, 2)
print(f"describe() : {t_desc:.2f} s | tasks cumulées : {taches_consommees():,}")

+-------+------------------+-----------+--------------+------------------+-----------------+---------------+--------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+-------------------+-----------------------------+
|summary|ID                |Case Number|Block         |IUCR              |Primary Type     |Description    |Location Description|Beat              |District          |Ward              |Community Area    |FBI Code          |X Coordinate      |Y Coordinate      |Year              |Latitude           |Longitude          |Location                     |
+-------+------------------+-----------+--------------+------------------+-----------------+---------------+--------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+---------------

describe() : 39.29 s | tasks cumulées : 258


In [21]:
t0 = time.perf_counter(); df.summary().show(truncate=False); t_sum = time.perf_counter() - t0
RESULTS["11_summary"] = round(t_sum, 2)
print(f"summary() : {t_sum:.2f} s | tasks cumulées : {taches_consommees():,}")

+-------+------------------+-----------+--------------+------------------+-----------------+---------------+--------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+-------------------+-----------------------------+
|summary|ID                |Case Number|Block         |IUCR              |Primary Type     |Description    |Location Description|Beat              |District          |Ward              |Community Area    |FBI Code          |X Coordinate      |Y Coordinate      |Year              |Latitude           |Longitude          |Location                     |
+-------+------------------+-----------+--------------+------------------+-----------------+---------------+--------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+---------------

### Conclusion Q11

`describe()` renvoie `count`, `mean`, `stddev`, `min` et `max`. `summary()` ajoute les quantiles
25, 50 et 75 % (`percentile_approx`), ce qui explique l'écart entre 39 s et 203 s.

Les deux relancent un scan complet, donc les enchaîner après un `cache()` est la seule façon de ne
pas payer deux fois.

## 12. Valeurs manquantes, valeurs uniques, effectifs par type de crime

In [22]:
# 1) valeurs manquantes : une seule passe, sans shuffle
t0 = time.perf_counter()
r_null = df.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in df.columns]).first()
t_null = time.perf_counter() - t0
RESULTS["12_manquantes"] = round(t_null, 2)
print(f"valeurs manquantes : {t_null:.2f} s (1 passe, 0 shuffle)")

# 2) valeurs uniques : countDistinct = un shuffle par colonne -> par lots de 5
t0 = time.perf_counter()
uniq = {}
for i in range(0, len(df.columns), 5):
    lot = df.columns[i:i+5]
    row = df.select([F.countDistinct(F.col(c)).alias(c) for c in lot]).first()
    uniq.update({c: row[c] for c in lot})
t_uniq = time.perf_counter() - t0
RESULTS["12_uniques"] = round(t_uniq, 2)
print(f"valeurs uniques   : {t_uniq:.2f} s (4 lots de countDistinct)\n")

print(f"{'colonne':<24}{'manquantes':>12}{'%':>8}{'uniques':>12}")
for c in df.columns:
    m, u = r_null[c], uniq[c]
    print(f"{c:<24}{m:>12,}{100*m/n:>7.2f}%{u:>12,}")

valeurs manquantes : 5.30 s (1 passe, 0 shuffle)


valeurs uniques   : 19.94 s (4 lots de countDistinct)

colonne                   manquantes       %     uniques
ID                                 0   0.00%   4,695,426
Case Number                        0   0.00%   4,695,409
Date                               0   0.00%   1,950,543
Block                              0   0.00%      42,382
IUCR                               0   0.00%         417
Primary Type                       0   0.00%          35
Description                        0   0.00%         554
Location Description          13,335   0.28%         168
Arrest                             0   0.00%           2
Domestic                           0   0.00%           2
Beat                               0   0.00%         306
District                          10   0.00%          25
Ward                          15,747   0.33%          51
Community Area                16,421   0.34%          78
FBI Code                           0   0.00%          27
X Coordinate                 139,

In [23]:
t0 = time.perf_counter()
crimes = (df.groupBy("Primary Type").count()
            .withColumn("pourcentage", F.round(F.col("count") / F.lit(n) * 100, 2))
            .orderBy(F.desc("count")))
crimes.show(50, truncate=False)
t_crimes = time.perf_counter() - t0
RESULTS["12_par_type"] = round(t_crimes, 2)
print(f"groupBy Primary Type : {t_crimes:.2f} s | total = {n:,} crimes")

+---------------------------------+-------+-----------+
|Primary Type                     |count  |pourcentage|
+---------------------------------+-------+-----------+
|THEFT                            |1044026|21.85      |
|BATTERY                          |856076 |17.91      |
|CRIMINAL DAMAGE                  |521203 |10.91      |
|NARCOTICS                        |450421 |9.43       |
|ASSAULT                          |304841 |6.38       |
|OTHER OFFENSE                    |295000 |6.17       |
|BURGLARY                         |269230 |5.63       |
|DECEPTIVE PRACTICE               |246498 |5.16       |
|MOTOR VEHICLE THEFT              |213950 |4.48       |
|ROBBERY                          |178236 |3.73       |
|CRIMINAL TRESPASS                |128985 |2.7        |
|WEAPONS VIOLATION                |63478  |1.33       |
|OFFENSE INVOLVING CHILDREN       |41131  |0.86       |
|PUBLIC PEACE VIOLATION           |35097  |0.73       |
|PROSTITUTION                     |34995  |0.73 

In [24]:
print(json.dumps(RESULTS, indent=2, ensure_ascii=False))
open("resultats.json", "w", encoding="utf-8").write(json.dumps(RESULTS, indent=2, ensure_ascii=False))

df.unpersist()
spark.stop()
print("\nSession Spark arrêtée.")

{
  "1_pandas": 1.66,
  "1_pandas_engine_c": 6.38,
  "2_spark_defaut_count": 1.93,
  "2_spark_header_count": 0.99,
  "3_partitions": 14,
  "3_colonnes": 22,
  "5_inferSchema_count": 6.91,
  "8_schema_manuel_count": 0.85,
  "9_limit_ms": 6.4,
  "9_head": 0.26,
  "9_take": 0.12,
  "9_collect_echec_s": 22,
  "10_cache_avant": 33.2,
  "10_cache_apres": 0.39,
  "11_describe": 39.29,
  "11_summary": 203.28,
  "12_manquantes": 5.3,
  "12_uniques": 19.94,
  "12_par_type": 1.09
}



Session Spark arrêtée.


---

### Note sur la variabilité

Les temps ci-dessus sont pris avec le fichier en cache disque. À froid, sur une première lecture
réelle, c'est 4 à 7 fois plus lent, et c'est ce que voit un poste qui ouvre le fichier pour la
première fois :

| cellule | à froid | en cache |
|---|---|---|
| `pd.read_csv` | 7,7 s | 1,7 s |
| `spark.read.csv(..., header=True).count()` | 6,7 s | 1,0 s |
| `inferSchema=True` + `count()` | 21,5 s | 6,9 s |

Sur l'export complet (1,9 Go, ~8,07 M lignes), comptez environ 1,7 fois ces valeurs. L'ordre de
grandeur et les conclusions ne changent pas.